Samantha Asefi (sma9am@virginia.edu)
DS 5001
8 May 2026

# Cleaning Data

The data that is collected from the fetchdata.ipynb is now cleaned in this file. A lot of the reusable methods were found and implemented using Claude's Sonnet 4.6. The point of this method is to go in and remove the notes section that was found at the bottom of the page and any unnecessary intro text that was at the tops of the pages as well.

## Imports

In [1]:
import pandas as pd
import numpy as np
from glob import glob
import re
import nltk
import os, re
from pathlib import Path
import re
from pathlib import Path
from nltk.corpus import stopwords
nltk.download('stopwords')

from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize, word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('tagsets')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package tagsets to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package tagsets is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_

True

## Mappings

The point of the mappings is to assign readable values to the books and chapters that are fetched from the internet. The files are currently being individually scraped "per page" and then recompiled to represent an organized book. Instead of using meta-data to parse this information out - due to the fact that there are many different works represented for the same the authors, it made it easier and more maintainable sense to do it this way. Basically, in the event I wanted to add more authors, which I debated for a balanced corpus, this would allow me to do that quite easily. 

In [2]:
AUTHOR_MAP = {
    'luxemburg_reform-or-revolution':          'Rosa Luxemburg',
    'mao_on-guerrilla-warfare':                'Mao Zedong',
    'mao_correcting-mistaken-ideas':           'Mao Zedong',
    'mao_tactics-against-japanese-imperialism':'Mao Zedong',
    'mao_on-practice':                         'Mao Zedong',
    'mao_win-the-masses':                      'Mao Zedong',
    'marx_communist-manifesto':                'Karl Marx & Friedrich Engels',
    'marx_wage-labour-and-capital':            'Karl Marx',
    'fourier_selections':                      'Charles Fourier'
}

TITLE_MAP = {
    'luxemburg_reform-or-revolution':          'Reform or Revolution',
    'mao_on-guerrilla-warfare':                'On Guerrilla Warfare',
    'mao_correcting-mistaken-ideas':           'On Correcting Mistaken Ideas in the Party',
    'mao_tactics-against-japanese-imperialism':'On Tactics Against Japanese Imperialism',
    'mao_on-practice':                         'On Practice',
    'mao_win-the-masses':                      'Win the Masses for the Anti-Japanese Front',
    'marx_communist-manifesto':                'The Communist Manifesto',
    'marx_wage-labour-and-capital':            'Wage Labour and Capital',
    'fourier_selections':                      'Selections from his Writings'
}

In [3]:
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

In [4]:
CORPUS_DIR = Path("corpus/books")

## Parsing Each Page

Based on the filename patterns that were found in the downloaded links/pages - I have regex patterns that will parse out what chapter of the book it is. This method is basically figuring out what chapter of the book each scraped page is and uses a series of checks to be able to parse that out and add it to the tables.

In [5]:
def parse_chap_num(filepath: Path):
    stem = filepath.stem.lower()
    
    m = re.match(r"ch(\d+)", stem)
    if m:
        return int(m.group(1))
    
    if re.match(r"mswv", stem):
        return 1
    
    if stem == "index":
        return 0
    
    return None

## Cleaning Out the Junk

Parses a single scraped chapter file back into structured data. It stops the parsing and organizing when it reaches the NOTES section which was pretty much present on every single one of the scraped pages.

In [6]:
def parse_raw_txt(filepath: Path):
    text = filepath.read_text(encoding="utf-8")
    soup = BeautifulSoup(text, "html.parser")
    
    source_url_tag = soup.find("source_url")
    page_title_tag = soup.find("page_title")
    chapter_title_tag = soup.find("chapter_title")
    
    source_url = source_url_tag.get_text(strip=True) if source_url_tag else None
    page_title = page_title_tag.get_text(strip=True) if page_title_tag else None
    chapter_title = chapter_title_tag.get_text(strip=True) if chapter_title_tag else None
    
    paras = []
    for p in soup.find_all("para"):
        para_num = int(p.get("id"))
        para_text = p.get_text(" ", strip=True)
        
        # stop parsing when we hit the NOTES section
        if re.match(r"^\s*NOTES\s*$", para_text, re.IGNORECASE):
            break
            
        paras.append((para_num, para_text))
    
    return {
        "source_url": source_url,
        "page_title": page_title,
        "chapter_title": chapter_title,
        "paragraphs": paras
    }

## Helper method to Check Content

This method was created to check whether or not the information that is being parsed is real or not. For example, any line that is too short (under 15 characters), was disregarded. Anything that begins with Source:, navigation language that is found on websites, the author's name and year patterns found randomly within the text, footer language etc. were all filtered out and are marked as not real content.

In [7]:
def is_real_content(text: str) -> bool:
    if not text:
        return False
    
    txt = text.strip()
    
    # too short to be real content (single words, page numbers, etc.)
    if len(txt) < 15:
        return False
    
    # source/bibliographic line
    if txt.startswith("Source:"):
        return False
    
    # archive/footer line
    if "Archive" in txt:
        return False
    
    # author lifespan line, e.g. Charles Fourier (1772-1837)
    if re.fullmatch(r".*\(\d{4}\s*-\s*\d{4}\)", txt):
        return False
    
    # navigation lines like "Next: Chap.2" or "Previous:"
    if re.match(r"^\s*(Next|Previous|Prev)\s*:", txt, re.IGNORECASE):
        return False
    
    # "Last updated on" footer
    if re.match(r"^\s*Last updated on", txt, re.IGNORECASE):
        return False
    
    # footnote/endnote lines — start with a number or bracketed number
    if re.match(r"^\s*(\d+\.|\[\d+\])", txt):
        return False
    
    if txt in TITLE_MAP.values():
        return False

    return True



# Register and Chunk

In [8]:
library_rows = []
doc_rows = []

for book_dir in sorted(CORPUS_DIR.iterdir()):
    if not book_dir.is_dir():
        continue

    book_id = book_dir.name

    for filepath in sorted(book_dir.glob("*.txt")):
        chap_num = parse_chap_num(filepath)
        parsed = parse_raw_txt(filepath)

        # LIB row = one chapter file
        library_rows.append({
            'book_id': book_id,
            'chap_num': chap_num,
            'author': AUTHOR_MAP.get(book_id),
            'book_title': TITLE_MAP.get(book_id),
            'filepath': str(filepath),
            'source_url': parsed['source_url'],
            'page_title': parsed['page_title'],
            'chapter_title': parsed['chapter_title'],
            'n_paragraphs_raw': len(parsed['paragraphs'])
        })

        # DOC rows = one paragraph per row
        for para_num, para_text in parsed['paragraphs']:
            if is_real_content(para_text):
                doc_rows.append({
                    'book_id': book_id,
                    'chap_num': chap_num,
                    'para_num': para_num,
                    'author': AUTHOR_MAP.get(book_id),
                    'book_title': TITLE_MAP.get(book_id),
                    'para_str': para_text
                })

LIB = pd.DataFrame(library_rows).sort_values(['book_id', 'chap_num']).reset_index(drop=True)
DOC = pd.DataFrame(doc_rows).sort_values(['book_id', 'chap_num', 'para_num']).reset_index(drop=True)

In [9]:
LIB.head()

,book_id,chap_num,author,book_title,filepath,source_url,page_title,chapter_title,n_paragraphs_raw
0,fourier_selections,1,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch01.txt,https://www.marxists.org/reference/archive/fou...,"Charles Fourier, Selections from his Writings",Of the Role of the Passions,41
1,fourier_selections,2,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch02.txt,https://www.marxists.org/reference/archive/fou...,"Charles Fourier, Selections from his Writings",Of Education,23
2,fourier_selections,3,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch03.txt,https://www.marxists.org/reference/archive/fou...,Selection from Charles Fourier,“Universal Harmony”,10
3,fourier_selections,4,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch04.txt,https://www.marxists.org/reference/archive/fou...,Selection from Charles Fourier,“Letter to the High Judge”,43
4,fourier_selections,5,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch05.txt,https://www.marxists.org/reference/archive/fou...,Selection from Charles Fourier,“Indices and Methods which led to the Discovery”,34


In [10]:
DOC = DOC.set_index(['book_id', 'chap_num', 'para_num']).sort_index()

In [11]:
DOC.head()

author  \
book_id            chap_num para_num                    
fourier_selections 1        3         Charles Fourier   
                            4         Charles Fourier   
                            5         Charles Fourier   
                            6         Charles Fourier   
                            7         Charles Fourier   

                                                        book_title  \
book_id            chap_num para_num                                 
fourier_selections 1        3         Selections from his Writings   
                            4         Selections from his Writings   
                            5         Selections from his Writings   
                            6         Selections from his Writings   
                            7         Selections from his Writings   

                                                                               para_str  
book_id            chap_num para_num                                                     
fourier_selections 1        3         All those philosophical whims called duties ha...  
                            4         The learned world is wholly imbued with a doct...  
                            5         Morality teaches man to be at war with himself...  
                            6         It is true that these impulses entice us only ...  
                            7         The passions, believed to be the enemies of co...

# Tokenize and Annotate

In [12]:
def tokenize(doc_df, OHCO=OHCO, remove_pos_tuple=False, ws=False):

    # Paragraphs -> Sentences
    df = (
        doc_df['para_str']
        .dropna()
        .pipe(lambda s: s[s.str.strip() != ''])
        .apply(lambda x: pd.Series(nltk.sent_tokenize(x)))
        .stack()
        .dropna()                                          # ← add here too
        .pipe(lambda s: s[s.str.strip() != ''])           # ← and here
        .to_frame(name='sent_str')
    )

    # Sentences -> Tokens
    def tokenize_sentence(x):
        if ws:
            toks = nltk.WhitespaceTokenizer().tokenize(x)
        else:
            toks = nltk.word_tokenize(x)
        return pd.Series(nltk.pos_tag(toks))

    df = (
        df['sent_str']
        .apply(tokenize_sentence)
        .stack()
        .dropna()                                          # ← add here too
        .to_frame(name='pos_tuple')
    )

    # Pull token/POS out of tuple
    df['token_str'] = df['pos_tuple'].apply(lambda x: x[0])
    df['pos'] = df['pos_tuple'].apply(lambda x: x[1])

    if remove_pos_tuple:
        df = df.drop(columns='pos_tuple')

    df.index.names = OHCO

    return df

In [13]:
TOKEN = tokenize(DOC, OHCO=OHCO, remove_pos_tuple=False, ws=False)


In [14]:
doc_meta = (
    DOC.reset_index()[['book_id', 'chap_num', 'author', 'book_title']]
    .drop_duplicates()
)

TOKEN = TOKEN.reset_index()

TOKEN['term_str'] = (
    TOKEN['token_str']
    .str.lower()
    .str.replace(r"[^a-z]+", "", regex=True
    )
)

TOKEN = TOKEN[TOKEN['term_str'].notna()].copy()
TOKEN = TOKEN[TOKEN['term_str'] != ""].copy()
TOKEN = TOKEN[TOKEN['term_str'].str.len() > 1].copy()

In [15]:
STOPWORDS = set(stopwords.words('english'))

TOKEN['is_stop'] = TOKEN['term_str'].isin(STOPWORDS)

# 5. (optional but recommended) remove stopwords
TOKEN = TOKEN[~TOKEN['is_stop']].copy()

# 6. restore OHCO index
TOKEN = TOKEN.set_index(OHCO).sort_index()

In [16]:
TOKEN.head()

pos_tuple  \
book_id            chap_num para_num sent_num token_num                        
fourier_selections 1        3        0        2          (philosophical, JJ)   
                                              3                 (whims, NNS)   
                                              4                (called, VBN)   
                                              5                (duties, NNS)   
                                              8               (relation, NN)   

                                                             token_str  pos  \
book_id            chap_num para_num sent_num token_num                       
fourier_selections 1        3        0        2          philosophical   JJ   
                                              3                  whims  NNS   
                                              4                 called  VBN   
                                              5                 duties  NNS   
                                              8               relation   NN   

                                                              term_str  \
book_id            chap_num para_num sent_num token_num                  
fourier_selections 1        3        0        2          philosophical   
                                              3                  whims   
                                              4                 called   
                                              5                 duties   
                                              8               relation   

                                                         is_stop  
book_id            chap_num para_num sent_num token_num           
fourier_selections 1        3        0        2            False  
                                              3            False  
                                              4            False  
                                              5            False  
                                              8            False

In [17]:
TOKEN[TOKEN.pos.str.match('^JJ')]

pos_tuple  \
book_id                      chap_num para_num sent_num token_num                        
fourier_selections           1        3        0        2          (philosophical, JJ)   
                                                        66            (invariable, JJ)   
                                      4        0        1                (learned, JJ)   
                                                        15                (mortal, JJ)   
                                                        18             (passional, JJ)   
...                                                                                ...   
marx_wage-labour-and-capital 9        21       1        23                 (whole, JJ)   
                                      22       0        16              (greater, JJR)   
                                                        29               (working, JJ)   
                                                        44                 (rapid, JJ)   
                                                        51             (favorable, JJ)   

                                                                       token_str  \
book_id                      chap_num para_num sent_num token_num                  
fourier_selections           1        3        0        2          philosophical   
                                                        66            invariable   
                                      4        0        1                learned   
                                                        15                mortal   
                                                        18             passional   
...                                                                          ...   
marx_wage-labour-and-capital 9        21       1        23                 whole   
                                      22       0        16               greater   
                                                        29               working   
                                                        44                 rapid   
                                                        51             favorable   

                                                                   pos  \
book_id                      chap_num para_num sent_num token_num        
fourier_selections           1        3        0        2           JJ   
                                                        66          JJ   
                                      4        0        1           JJ   
                                                        15          JJ   
                                                        18          JJ   
...                                                                ...   
marx_wage-labour-and-capital 9        21       1        23          JJ   
                                      22       0        16         JJR   
                                                        29          JJ   
                                                        44          JJ   
                                                        51          JJ   

                                                                        term_str  \
book_id                      chap_num para_num sent_num token_num                  
fourier_selections           1        3        0        2          philosophical   
                                                        66            invariable   
                                      4        0        1                learned   
                                                        15                mortal   
                                                        18             passional   
...                                                                          ...   
marx_wage-labour-and-capital 9        21       1        23                 whole   
                                      22       0        16               greater   
          

## Building Vocab Table

Based on the Token table - the vocab table is created here. We have added POS tags for all of the different words, how many times a word appears in the text, a num flag to identify whether or not the word is a number or not.

In [18]:
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))
ps = PorterStemmer()

tok = TOKEN.reset_index()

pos_max = (
    tok.groupby('term_str')['pos']
    .agg(lambda x: x.value_counts().idxmax())
    .rename('pos_max')
)

VOCAB = (
    tok
    .groupby('term_str', dropna=True)
    .agg(n=('term_str', 'size'))
    .join(pos_max)
    .reset_index()
    .sort_values('term_str')
    .reset_index(drop=True)
)

VOCAB.index.name = 'term_id'
VOCAB['num'] = VOCAB['term_str'].str.isnumeric().fillna(False).astype(int)
VOCAB['stop'] = VOCAB['term_str'].isin(STOPWORDS).astype(int)
VOCAB['p_stem'] = VOCAB['term_str'].apply(lambda x: ps.stem(x) if pd.notna(x) else np.nan)

VOCAB.head()


,term_str,n,pos_max,num,stop,p_stem
term_id,,,,,,
0,aback,34,NN,0,0,aback
1,abandon,36,VB,0,0,abandon
2,abandoned,34,VBN,0,0,abandon
3,abandoning,25,VBG,0,0,abandon
4,abandonment,3,NN,0,0,abandon


In [19]:
OUTDIR = Path("corpus/f2")
OUTDIR.mkdir(parents=True, exist_ok=True)

In [20]:
LIB.to_csv("corpus/f2/LIB.csv", index=False)
DOC.reset_index().to_csv("corpus/f2/DOC.csv", index=False)
TOKEN.reset_index().to_csv("corpus/f2/TOKEN.csv", index=False)
VOCAB.reset_index().to_csv("corpus/f2/VOCAB.csv", index=False)